In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub')
    
    if not os.path.exists("models/benchmark/ConLID/repo"):
        print("Setting up ConLID dependencies...")
        os.makedirs("models/benchmark/ConLID", exist_ok=True)
        os.system('git clone https://github.com/epfl-nlp/language-identification.git models/benchmark/ConLID/repo')
        os.system('pip install -q -r models/benchmark/ConLID/repo/requirements.txt')
        
    print("Setup complete!")


In [2]:
# NOTE: If running this notebook manually in the IDE, make sure to select the `conlid-venv` kernel!
input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_results'

In [3]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

TARGET_LANGUAGES = {
    "eng": "eng_Latn",  # English (baseline)
    "sin": "sin_Sinh",  # Sinhala
    "san": "san_Deva",  # Sanskrit
    "tam": "tam_Taml",  # Tamil
    "hin": "hin_Deva",  # Hindi
    "ben": "ben_Beng",  # Bengali
    "arb": "arb_Arab",  # Arabic (Modern Standard)
    "fra": "fra_Latn",  # French
    "deu": "deu_Latn",  # German
}

def load_dataset(file_path):
    print(f"\nLoading {os.path.basename(file_path)}...")
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("label") in TARGET_LANGUAGES:
                records.append(row)
    df = pd.DataFrame(records)
    if not df.empty:
        df["flores_label"] = df["label"].map(TARGET_LANGUAGES)
        print(f"Loaded {len(df)} rows across {df['label'].nunique()} target languages")
    else:
        print("No matching target languages found in this dataset.")
    return df

def evaluate_and_save(results, model_name, dataset_name, target_labels):
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(
        results["true_label"], results["predicted_label"],
        average="macro", labels=target_labels,
    )

    print("\n" + "=" * 48)
    print(f"ZERO-SHOT BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(classification_report(
        results["true_label"], results["predicted_label"],
        labels=target_labels, digits=4,
    ))

    os.makedirs(output_dir, exist_ok=True)
    out_file = os.path.join(output_dir, f"{model_name.replace(' ', '_').replace('-', '_').lower()}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")
    return results

dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
if not dataset_files:
    print(f"No datasets found in {input_dir}.")


In [4]:
import os
import sys

REPO_DIR = "models/benchmark/ConLID/repo"

if not os.path.exists(REPO_DIR):
    raise RuntimeError(f"{REPO_DIR} not found. Please run 'make setup-conlid' in the pipeline root first.")

sys.path.append(REPO_DIR)
from model import ConLID  # noqa: E402
from huggingface_hub import snapshot_download  # noqa: E402
from tqdm.auto import tqdm  # noqa: E402

print("Downloading ConLID checkpoints...")
checkpoint_dir = os.path.join(REPO_DIR, "checkpoints", "conlid")
snapshot_download(repo_id="epfl-nlp/ConLID", local_dir=checkpoint_dir)

print("Loading ConLID model...")
conlid_model = ConLID.from_pretrained(dir=checkpoint_dir)
model_name = "ConLID"
target_labels = sorted(set(TARGET_LANGUAGES.values()))

for file_path in dataset_files:
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    df = load_dataset(file_path)
    if df.empty: continue
    
    texts = df["text"].astype(str).tolist()
    print(f"Evaluating {len(texts)} samples with {model_name}...")
    
    predicted_labels = []
    for text in tqdm(texts):
        pred_result = conlid_model.predict(text, k=1)
        predicted_labels.append(pred_result[0][0])

    results = df[["text", "label", "source"]].copy()
    results["true_label"] = df["flores_label"]
    results["predicted_label"] = predicted_labels

    evaluate_and_save(results, model_name, dataset_name, target_labels)


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 53.06it/s]


Loading ConLID model...

Loading flores_plus.jsonl...
Loaded 12116 rows across 9 target languages
Evaluating 12116 samples with ConLID...


100%|██████████| 12116/12116 [00:18<00:00, 659.30it/s] 
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/De


ZERO-SHOT BENCHMARK RESULTS (ConLID on flores_plus)
Accuracy:  75.52%
Macro F1:  79.51%

Per-language breakdown:

              precision    recall  f1-score   support

    arb_Arab     0.9957    0.2297    0.3733      2024
    ben_Beng     1.0000    0.9990    0.9995      1012
    deu_Latn     1.0000    0.9941    0.9970      1012
    eng_Latn     1.0000    0.9921    0.9960      1012
    fra_Latn     1.0000    1.0000    1.0000      1012
    hin_Deva     1.0000    0.9970    0.9985      1012
    san_Deva     0.0000    0.0000    0.0000      1327
    sin_Sinh     0.6654    0.9770    0.7916      2693
    tam_Taml     0.9990    1.0000    0.9995      1012

   micro avg     0.8734    0.7552    0.8100     12116
   macro avg     0.8511    0.7988    0.7951     12116
weighted avg     0.8153    0.7552    0.7387     12116


Saved predictions to datasets/benchmark_results/conlid_flores_plus.csv


Loading commonlid.jsonl...
Loaded 74052 rows across 9 target languages
Evaluating 74052 samples with ConLI

100%|██████████| 74052/74052 [01:20<00:00, 922.60it/s] 



ZERO-SHOT BENCHMARK RESULTS (ConLID on commonlid)
Accuracy:  71.83%
Macro F1:  78.77%

Per-language breakdown:

              precision    recall  f1-score   support

    arb_Arab     0.9997    0.6213    0.7664     26152
    ben_Beng     1.0000    0.9258    0.9615      1886
    deu_Latn     0.9918    0.8160    0.8953      7553
    eng_Latn     0.9964    0.7411    0.8500     27461
    fra_Latn     0.9799    0.8277    0.8974      3233
    hin_Deva     0.9958    0.8991    0.9450      3666
    san_Deva     0.0000    0.0000    0.0000      1327
    sin_Sinh     0.6654    0.9770    0.7916      2693
    tam_Taml     0.9643    1.0000    0.9818        81

   micro avg     0.9718    0.7183    0.8260     74052
   macro avg     0.8437    0.7564    0.7877     74052
weighted avg     0.9665    0.7183    0.8175     74052


Saved predictions to datasets/benchmark_results/conlid_commonlid.csv


Loading wili-2018.jsonl...
Loaded 10020 rows across 8 target languages
Evaluating 10020 samples with ConLID...

100%|██████████| 10020/10020 [00:27<00:00, 366.09it/s]
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/pro


ZERO-SHOT BENCHMARK RESULTS (ConLID on wili-2018)
Accuracy:  83.74%
Macro F1:  73.56%

Per-language breakdown:

              precision    recall  f1-score   support

    arb_Arab     0.0000    0.0000    0.0000         0
    ben_Beng     1.0000    0.8930    0.9435      1000
    deu_Latn     0.9989    0.9440    0.9707      1000
    eng_Latn     0.9224    0.9750    0.9480      1000
    fra_Latn     0.9869    0.9780    0.9824      1000
    hin_Deva     1.0000    0.9800    0.9899      1000
    san_Deva     0.0000    0.0000    0.0000      1327
    sin_Sinh     0.6654    0.9770    0.7916      2693
    tam_Taml     0.9990    0.9900    0.9945      1000

   micro avg     0.8551    0.8374    0.8462     10020
   macro avg     0.7303    0.7486    0.7356     10020
weighted avg     0.7684    0.8374    0.7945     10020


Saved predictions to datasets/benchmark_results/conlid_wili-2018.csv

